In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
class RoPE(nn.Module):
    def __init__(self, dim, theta=10000.0):
        super().__init__()
        self.dim = dim
        self.theta = theta
        # self.freq_cos = 0
        # self.freq_sin = 0
    
    def precompute_freq(self, end):
        freq = 1.0/ (self.theta ** (torch.arange(0,self.dim,2)[: (self.dim//2)].float() / self.dim))
        t = torch.arange(end, device=freq.device)
        freq = torch.outer(t, freq).float()
        freq = torch.cat((freq, freq), dim = -1)
        return freq.cos(), freq.sin()
    
    def rotary_emd(self, x, freq_cos, freq_sin):
        Batch, Time_step, dim_head = x.shape
        half_dim = dim_head // 2
        x1 = x[..., : half_dim]
        x2 = x[..., half_dim: ]

        cos = freq_cos[: Time_step, :half_dim]
        sin = freq_sin[: Time_step, :half_dim]

        output1 = (x1 * cos) - (x2 * sin)
        output2 = (x1 * sin) + (x2 * cos)
        return torch.cat((output1, output2), dim=-1)
